0. 준비: 라이브러리 설치

In [ ]:
pip install cryptography

1. RSA 공개키/개인키 생성 코드

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

def generate_keys():
    # 2048비트 RSA 키 생성
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048,
    )

    public_key = private_key.public_key()

    # 개인키를 PEM 형식으로 저장 (비밀번호 없이 예시)
    with open("private_key.pem", "wb") as f:
        f.write(
            private_key.private_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PrivateFormat.PKCS8,
                encryption_algorithm=serialization.NoEncryption(),
            )
        )

    # 공개키를 PEM 형식으로 저장
    with open("public_key.pem", "wb") as f:
        f.write(
            public_key.public_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PublicFormat.SubjectPublicKeyInfo,
            )
        )

    print("키 생성 완료: private_key.pem, public_key.pem")

키 생성 완료: private_key.pem, public_key.pem


2. 공개키 전달

In [4]:
with open("public_key.pem", "r") as f:
    public_key_text = f.read()

public_key_text

'-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAogS0pkbBLP+vS5CUqGtc\nlGvxj2qw/4IXFJa4GZTHftyCxTR3+qkCI0JRkw+KKDfA9VZr9W2syO4yarbxM/0L\nf+HeGDka8LEdg/A0rvb6RhbqyKSQe9d7ebjbffbxi4YJIlI6qrLvdttsNrsorsTX\nzWlUUzOrp8ffCBZ1lzI5lwalxdxpqQzD48er3w7U/Qfot2WNw3c6ynmBJW/SjhNK\nwQZ1J8dpG33pjKx6ThwfLlR8kjtpeO2XiWjbhxU38sKg0TbYP8NH19okP6afFtiG\nYb+OfI5+Ie7/0MuVwLR2+X0vnsOHbhtnZrgG+46+PgaHi0QnPnwLt1ZoxYVo2FmU\nbQIDAQAB\n-----END PUBLIC KEY-----\n'

3.1. 파일 전송

In [ ]:
# 서버가 암호화해서 전송할 파일 경로
path = "secret_data.txt"

In [ ]:
import socket
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding

# 서버 실행
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.bind(("0.0.0.0", 5000))
server.listen(1)

print("서버 대기중...")

conn, addr = server.accept()
print(f"클라이언트 접속: {addr}")

# ================================
# 1) 클라이언트 공개키 수신
# - 먼저 길이를 4바이트 정수로 받는 방식 사용
# ================================
import struct

# 공개키 길이 4바이트 수신
key_len_bytes = conn.recv(4)
key_len = struct.unpack(">I", key_len_bytes)[0]

# 실제 공개키 데이터 수신
client_public_key_bytes = conn.recv(key_len)

# 공개키 객체로 로드
client_public_key = serialization.load_pem_public_key(client_public_key_bytes)

print("클라이언트 공개키 수신 완료")

# ================================
# 2) 서버 파일 읽기
# ================================
with open(path, "rb") as f:
    file_data = f.read()

# ================================
# 3) 파일을 클라이언트 공개키로 암호화
# ================================
encrypted_data = client_public_key.encrypt(
    file_data,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)

print(f"파일 암호화 완료 ({len(encrypted_data)} bytes)")

# ================================
# 4) 암호문을 길이 + 본문 형태로 전송
# ================================
conn.sendall(struct.pack(">I", len(encrypted_data)))
conn.sendall(encrypted_data)

print("암호문 전송 완료")
conn.close()
server.close()


서버 대기중...
클라이언트 접속: ('192.168.75.62', 56370)
복호화된 데이터: hello secret key


3.1. 파일 수신

In [ ]:
SERVER_IP = ""

In [ ]:
import socket
import struct
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import serialization, hashes

# ==========================
# 접속할 서버 IP 입력
# ==========================
SERVER_IP = ""   # ← 여기에 직접 입력하세요
PORT = 5000

# ==========================
# 1. 클라이언트 RSA 키 생성
# ==========================
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

public_key = private_key.public_key()

# 공개키를 PEM 형식으로 직렬화
public_pem = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
)

# PEM 길이 계산
public_key_len = len(public_pem)

# ==========================
# 2. 서버 접속
# ==========================
client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client.connect((SERVER_IP, PORT))
print("서버 접속 완료")

# ==========================
# 3. 공개키 길이 + 공개키 전송
# ==========================
client.sendall(struct.pack(">I", public_key_len))  # 4바이트 길이
client.sendall(public_pem)

print("클라이언트 공개키 전송 완료")

# ==========================
# 4. 서버에서 암호문 수신
# ==========================

# 암호문 길이 수신
encrypted_len_bytes = client.recv(4)
encrypted_len = struct.unpack(">I", encrypted_len_bytes)[0]

# 암호문 본문 수신
encrypted_data = client.recv(encrypted_len)
print(f"암호문 수신 완료 ({encrypted_len} bytes)")

client.close()

# ==========================
# 5. 개인키로 복호화
# ==========================
plaintext = private_key.decrypt(
    encrypted_data,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)

print("복호화 완료!")

# ==========================
# 6. 파일로 저장
# ==========================
output_file = "received_decrypted_file.txt"
with open(output_file, "wb") as f:
    f.write(plaintext)

print(f"복호화된 파일 저장 완료 → {output_file}")
